In [1]:
import os

from click import prompt
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
load_dotenv('conf/.env',override=True)

True

In [2]:
!pip install langchainhub -q

In [3]:
from langchain.prompts import PromptTemplate
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

from langchain.tools import DuckDuckGoSearchRun
from langchain.tools import DuckDuckGoSearchResults
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper

from langchain import hub
from langchain.agents import Tool, AgentExecutor,initialize_agent,create_react_agent


In [4]:
llm = ChatGoogleGenerativeAI(model='gemini-1.5-flash',temperature=0)

In [5]:
template = '''
Answer the following questions to the best of your ability
Questions: 
{q}
'''
prompt_template=PromptTemplate(template=template)

In [7]:
prompt=hub.pull('hwchase17/react')
print(type(prompt))
print(prompt)

<class 'langchain_core.prompts.prompt.PromptTemplate'>
input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'] input_types={} partial_variables={} metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'} template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}'


C:\IDE\PycharmProjects\Langchain\venv\lib\site-packages\langsmith\client.py:261: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [8]:
print(prompt.input_variables)

['agent_scratchpad', 'input', 'tool_names', 'tools']


In [9]:
print(prompt.template)

Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}


In [11]:
print(prompt.output_schema)

<class 'langchain_core.prompts.prompt.PromptTemplateOutput'>


Building python repl tool object

In [12]:
from langchain_experimental.tools.python.tool import PythonREPLTool
python_repl =PythonREPLTool()
##check how a tool is differnt from agent. In last example you created an agent object using repltool 
python_repl_tool=Tool(
    name='python_repl',
    func=python_repl.run,
    description='Useful when you want the generative model to execute a python code to respond to the query '
)

Create a Wikipedia tool

In [13]:
api_wrapper=WikipediaAPIWrapper()
wikipedia=WikipediaQueryRun(api_wrapper=api_wrapper)
wikipedia_tool=Tool(
    name='wikipedia',
    func=wikipedia.run,
    description='Useful for when you need to lookup an article on wikipedia'
)


DuckDuckGo Search Engine

In [14]:
search = DuckDuckGoSearchRun()
duckduckgo_tool=Tool(
    name='DuckDuckGo Search',
    func=search.run,
    description='Useful for when you need to search anything on internet'
)

Initialize the tools

In [16]:
tools=[python_repl_tool,wikipedia_tool,duckduckgo_tool]
agent=create_react_agent(llm,tools,prompt)
agent_executor=AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=10
)

In [17]:
question='Generate the first 20 numbers in the Fibonacci series'
output = agent_executor.invoke({
    'input': prompt_template.format(q=question)
})



> Entering new AgentExecutor chain...


Python REPL can execute arbitrary code. Use with caution.


Thought:I need to generate the first 20 Fibonacci numbers.  I can do this using a Python loop.

Action: python_repl

Action Input:
```python
def fibonacci(n):
  if n <= 0:
    return []
  elif n == 1:
    return [0]
  else:
    list_fib = [0, 1]
    while len(list_fib) < n:
      next_fib = list_fib[-1] + list_fib[-2]
      list_fib.append(next_fib)
    return list_fib

print(fibonacci(20))
```
[0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377, 610, 987, 1597, 2584, 4181]
Thought:I now know the final answer.

Final Answer: The first 20 numbers in the Fibonacci series are: [0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377, 610, 987, 1597, 2584, 4181]


> Finished chain.


In [18]:
print(type(output))
print(output)

<class 'dict'>
{'input': '\nAnswer the following questions to the best of your ability\nQuestions: \nGenerate the first 20 numbers in the Fibonacci series\n', 'output': 'The first 20 numbers in the Fibonacci series are: [0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377, 610, 987, 1597, 2584, 4181]'}


In [19]:
question_ddg='Who is the current prime minister of Canada'
output_ddg = agent_executor.invoke({
    'input': prompt_template.format(q=question_ddg)
})



> Entering new AgentExecutor chain...
Thought:I need to find the current Prime Minister of Canada.  A quick search on Wikipedia should suffice.

Action: wikipedia
Action Input: Current Prime Minister of Canada
Page: Prime Minister of Canada
Summary: The prime minister of Canada (French: premier ministre du Canada) is the head of government of Canada. Under the Westminster system, the prime minister governs with the confidence of a majority of the elected House of Commons; as such, the prime minister typically sits as a member of Parliament (MP) and leads the largest party or a coalition of parties. As first minister, the prime minister selects ministers to form the Cabinet.
Not outlined in any constitutional document, the prime minister is appointed by the monarch's representative, the governor general, and the office exists per long-established convention. Constitutionally, executive authority is vested in the monarch (who is the head of state), but the powers of the monarch and gov

In [21]:
question_ddg2='Who is the current finance minister of Canada and why did the previous one resign. Is there a controversy'
output_ddg2 = agent_executor.invoke({
    'input': prompt_template.format(q=question_ddg2)
})



> Entering new AgentExecutor chain...
Thought:I need to find information about Canada's current finance minister and the circumstances surrounding the previous minister's resignation.  A web search should provide this information efficiently.

Action: DuckDuckGo Search
Action Input: "current Canadian finance minister" AND "previous Canadian finance minister resignation"


C:\IDE\PycharmProjects\Langchain\venv\lib\site-packages\langchain_community\utilities\duckduckgo_search.py:64: UserWarning: 'api' backend is deprecated, using backend='auto'
  ddgs_gen = ddgs.text(


No good DuckDuckGo Search Result was foundThought:I need to try a different search strategy. Let's try searching for each question separately.

Action: DuckDuckGo Search
Action Input: "current Canadian finance minister"


C:\IDE\PycharmProjects\Langchain\venv\lib\site-packages\langchain_community\utilities\duckduckgo_search.py:64: UserWarning: 'api' backend is deprecated, using backend='auto'
  ddgs_gen = ddgs.text(


The minister of finance (French: ministre des Finances) is the minister of the Crown in the Canadian Cabinet, who is responsible for overseeing the Department of Finance and presenting the federal government's budget each year. It is one of the most important positions in the Cabinet. Dominic LeBlanc is the 41st and current finance minister, assuming the role December 16, 2024 following the ... Prime Minister Justin Trudeau congratulates Dominic LeBlanc after he was sworn in as finance minister at a swearing in ceremony at Rideau Hall, in Ottawa, Monday, Dec. 16, 2024. Photo by Justin ... When the party won re-election in 2019 but as a minority government, she was named deputy prime minister and became Canada’s first female federal finance minister in 2020 after Bill Morneau ... Ministry (Cabinet) as of Friday, December 20, 2024 - Members of Parliament - House of Commons of Canada ... Minister of Canadian Heritage. Brome—Missisquoi. Quebec. The Honourable. ... Minister of Finance. Beau

C:\IDE\PycharmProjects\Langchain\venv\lib\site-packages\langchain_community\utilities\duckduckgo_search.py:64: UserWarning: 'api' backend is deprecated, using backend='auto'
  ddgs_gen = ddgs.text(


No good DuckDuckGo Search Result was foundThought:Let's try Wikipedia to see if it has more reliable and detailed information.

Action: wikipedia
Action Input: "Chrystia Freeland"
Page: Chrystia Freeland
Summary: Christina Alexandra Freeland  (born August 2, 1968) is a Canadian politician and journalist who has served as the member of Parliament (MP) for University—Rosedale since 2015. She previously served as the tenth deputy prime minister of Canada from 2019 to 2024. A member of the Liberal Party, she was first elected to the House of Commons of Canada in a by-election in 2013. First appointed to the Cabinet following the 2015 federal election, she has served in various posts including as the minister of finance from 2020 until her resignation from the 29th Canadian Ministry in 2024.
Born in Peace River, Alberta, Freeland completed a bachelor's degree at Harvard University, studying Russian history and literature before earning a master's degree in Slavonic studies from the Universi

In [22]:
print(output_ddg)

{'input': '\nAnswer the following questions to the best of your ability\nQuestions: \nWho is the current prime minister of Canada\n', 'output': 'Justin Trudeau'}


In [25]:
template = '''
Answer the following questions in german, to the best of your ability
Questions: 
{q}
'''
prompt_template=PromptTemplate(template=template)
prompt=hub.pull('hwchase17/react')
print(type(prompt))
print(prompt)


C:\IDE\PycharmProjects\Langchain\venv\lib\site-packages\langsmith\client.py:261: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


<class 'langchain_core.prompts.prompt.PromptTemplate'>
input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'] input_types={} partial_variables={} metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'} template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}'


In [26]:
question_ddg2_german='Who is the current finance minister of Canada and why did the previous one resign. Is there a controversy'
output_ddg2_german = agent_executor.invoke({
    'input': prompt_template.format(q=question_ddg2_german)
})



> Entering new AgentExecutor chain...
Thought:I need to find information about Canada's current and previous finance ministers and any controversies surrounding the resignation of the previous one.  I'll use DuckDuckGo to search for this information in English, then translate the relevant parts into German.

Action: DuckDuckGo Search
Action Input: "Canada finance minister resignation controversy"


C:\IDE\PycharmProjects\Langchain\venv\lib\site-packages\langchain_community\utilities\duckduckgo_search.py:64: UserWarning: 'api' backend is deprecated, using backend='auto'
  ddgs_gen = ddgs.text(


Canada's Finance Minister Chrystia Freeland has resigned from her post, citing disagreements with Prime Minister Justin Trudeau on how to respond to incoming President Donald Trump's threat of ... Chrystia Freeland's resignation as finance minister and deputy prime minister sent shock waves through Ottawa on Monday. "Chrystia Freeland is a good friend, someone I work with very closely … TORONTO (AP) — Canadian Prime Minister Justin Trudeau faced the biggest test of his political career after Finance Minister Chrystia Freeland, long one of his most powerful and loyal ministers, resigned from the Cabinet on Monday.. The stunning move raised questions about how much longer the prime minister of nearly 10 years — whose popularity has plummeted due to concerns about inflation ... Item 1 of 5 Canada's Deputy Prime Minister and Minister of Finance Chrystia Freeland speaks during Question Period in the House of Commons on Parliament Hill in Ottawa, Ontario, Canada December 3 ... Canada's Finan

C:\IDE\PycharmProjects\Langchain\venv\lib\site-packages\langchain_community\utilities\duckduckgo_search.py:64: UserWarning: 'api' backend is deprecated, using backend='auto'
  ddgs_gen = ddgs.text(


No good DuckDuckGo Search Result was foundThought:I will try a different approach. I'll use DuckDuckGo to search for the current Canadian finance minister, and then separately search for information on the previous minister's resignation.

Action: DuckDuckGo Search
Action Input: "current Canadian finance minister"


C:\IDE\PycharmProjects\Langchain\venv\lib\site-packages\langchain_community\utilities\duckduckgo_search.py:64: UserWarning: 'api' backend is deprecated, using backend='auto'
  ddgs_gen = ddgs.text(


The minister of finance (French: ministre des Finances) is the minister of the Crown in the Canadian Cabinet, who is responsible for overseeing the Department of Finance and presenting the federal government's budget each year. It is one of the most important positions in the Cabinet. Dominic LeBlanc is the 41st and current finance minister, assuming the role December 16, 2024 following the ... Prime Minister Justin Trudeau congratulates Dominic LeBlanc after he was sworn in as finance minister at a swearing in ceremony at Rideau Hall, in Ottawa, Monday, Dec. 16, 2024. Photo by Justin ... When the party won re-election in 2019 but as a minority government, she was named deputy prime minister and became Canada’s first female federal finance minister in 2020 after Bill Morneau ... Ministry (Cabinet) as of Friday, December 20, 2024 - Members of Parliament - House of Commons of Canada ... Minister of Canadian Heritage. Brome—Missisquoi. Quebec. The Honourable. ... Minister of Finance. Beau

C:\IDE\PycharmProjects\Langchain\venv\lib\site-packages\langchain_community\utilities\duckduckgo_search.py:64: UserWarning: 'api' backend is deprecated, using backend='auto'
  ddgs_gen = ddgs.text(


Ms. Freeland’s sudden resignation left the government scrambling to figure out how to handle the unveiling of its fall economic statement, pushing back the media embargo period from 10 a.m. to 1 ... After Freeland's resignation six days later on December 16, Poilievre claimed Trudeau had "lost control, yet clings to power." [ 39 ] Bloc Québécois leader Yves-François Blanchet , whose party had already voted alongside the Conservatives to oust the Trudeau government in the last of multiple unsuccessful motions of non-confidence, remarked on December 16 that "the Trudeau government is done." All you need to know about Chrystia Freeland and why she resigned from Trudeau's cabinet In 2019 she was named deputy prime minister, and in 2020 she also took on the role of finance minister. She ... Chrystia Freeland's resignation as finance minister and deputy prime minister sent shock waves through Ottawa on Monday. "Chrystia Freeland is a good friend, someone I work with very closely … The shock 